In [ ]:
from collections.abc import Sequence
from sklearn import preprocessing
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shutil
import os
from sklearn.metrics import roc_curve, auc



# Encode text values to dummy variables(i.e. [1,0,0],[0,1,0],[0,0,1] for red,green,blue)
def encode_text_dummy(df, name):
    dummies = pd.get_dummies(df[name])
    for x in dummies.columns:
        dummy_name = "{}-{}".format(name, x)
        df[dummy_name] = dummies[x]
    df.drop(name, axis=1, inplace=True)


# Encode text values to indexes(i.e. [1],[2],[3] for red,green,blue).
def encode_text_index(df, name):
    le = preprocessing.LabelEncoder()
    df[name] = le.fit_transform(df[name])
    return le.classes_


# Encode a numeric column as zscores
def encode_numeric_zscore(df, name, mean=None, sd=None):
    if mean is None:
        mean = df[name].mean()

    if sd is None:
        sd = df[name].std()

    df[name] = (df[name] - mean) / sd


# Convert all missing values in the specified column to the median
def missing_median(df, name):
    med = df[name].median()
    df[name] = df[name].fillna(med)


# Convert all missing values in the specified column to the default
def missing_default(df, name, default_value):
    df[name] = df[name].fillna(default_value)


# Convert a Pandas dataframe to the x,y inputs that TensorFlow needs
def to_xy(df, target):
    result = []
    for x in df.columns:
        if x != target:
            result.append(x)
    # find out the type of the target column. 
    target_type = df[target].dtypes
    target_type = target_type[0] if isinstance(target_type, Sequence) else target_type
    # Encode to int for classification, float otherwise. TensorFlow likes 32 bits.
    if target_type in (np.int64, np.int32):
        # Classification
        dummies = pd.get_dummies(df[target])
        return df[result].values.astype(np.float32), dummies.values.astype(np.float32)
    else:
        # Regression
        return df[result].values.astype(np.float32), df[target].values.astype(np.float32)

# Nicely formatted time string
def hms_string(sec_elapsed):
    h = int(sec_elapsed / (60 * 60))
    m = int((sec_elapsed % (60 * 60)) / 60)
    s = sec_elapsed % 60
    return "{}:{:>02}:{:>05.2f}".format(h, m, s)


# Regression chart.
def chart_regression(pred,y,sort=True):
    t = pd.DataFrame({'pred' : pred, 'y' : y.flatten()})
    if sort:
        t.sort_values(by=['y'],inplace=True)
    a = plt.plot(t['y'].tolist(),label='expected')
    b = plt.plot(t['pred'].tolist(),label='prediction')
    plt.ylabel('output')
    plt.legend()
    plt.show()

# Remove all rows where the specified column is +/- sd standard deviations
def remove_outliers(df, name, sd):
    drop_rows = df.index[(np.abs(df[name] - df[name].mean()) >= (sd * df[name].std()))]
    df.drop(drop_rows, axis=0, inplace=True)


# Encode a column to a range between normalized_low and normalized_high.
def encode_numeric_range(df, name, normalized_low=-1, normalized_high=1,
                         data_low=None, data_high=None):
    if data_low is None:
        data_low = min(df[name])
        data_high = max(df[name])

    df[name] = ((df[name] - data_low) / (data_high - data_low)) \
               * (normalized_high - normalized_low) + normalized_low
    
def plot_confusion_matrix(cm, names, title='Confusion matrix', cmap=plt.cm.Blues):
    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(names))
    plt.xticks(tick_marks, names, rotation=45)
    plt.yticks(tick_marks, names)
    plt.tight_layout()
    plt.ylabel('True label')
    plt.xlabel('Predicted label')

def plot_roc(pred,y):
    fpr, tpr, thresholds = roc_curve(y, pred)
    roc_auc = auc(fpr, tpr)

    plt.figure()
    plt.plot(fpr, tpr, label='ROC curve (area = %0.2f)' % roc_auc)
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Receiver Operating Characteristic (ROC)')
    plt.legend(loc="lower right")
    plt.show()


In [ ]:
X_train = pd.read_csv('dataset/cleaned_dataset/train_feature_analysis.csv')
X_test = pd.read_csv('dataset/cleaned_dataset/test_feature_analysis.csv')

X_train.shape, X_test.shape

In [ ]:
y_train = X_train['label']
y_test = X_test['label']

y_train.shape, y_test.shape

In [ ]:
X_train.columns

In [ ]:
X_train.drop(columns=['id', 'label'], inplace=True)
X_test.drop(columns=['id', 'label'], inplace=True)

In [ ]:
X_train.shape, X_test.shape

In [ ]:
X_train_pad = np.pad(X_train.values, 
                      ((0, 0), (0, 14*14 - len(X_train.columns))),
                      mode='constant',
                      constant_values=0)

X_test_pad = np.pad(X_test.values, 
                      ((0, 0), (0, 14*14 - len(X_train.columns))),
                      mode='constant',
                      constant_values=0)

X_train_pad.shape, X_test_pad.shape

In [ ]:

# reshaped for larger datasets 

X_train_reshaped = X_train_pad.reshape(-1, 14, 14, 1)
X_test_reshaped = X_test_pad.reshape(-1, 14, 14, 1)

X_train_reshaped.shape, X_test_reshaped.shape


In [ ]:
BATCH_SIZE=64
EPOCHS=50
LEARNING_RATE=0.001

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Flatten, Conv2D, MaxPooling2D
from tensorflow.keras.optimizers import Adam, SGD
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

from sklearn.metrics import f1_score
from sklearn import metrics
from sklearn.metrics import confusion_matrix
 
input_shape = (14, 14, 1)

model = Sequential()

model.add(Conv2D(64, kernel_size=(3,3), strides=(1,1), padding='same', activation='relu', input_shape=input_shape))
model.add(Conv2D(64, (3, 3), activation='relu', padding='same'))
model.add(MaxPooling2D(pool_size=(2, 2), strides=2))

model.add(Conv2D(128, (3, 3), activation='relu', padding='same'))
model.add(Conv2D(128, (3, 3), activation='relu', padding='same'))
model.add(MaxPooling2D(pool_size=(2, 2), strides=2))

model.add(Conv2D(256, (3, 3), activation='relu', padding='same'))
model.add(Conv2D(256, (3, 3), activation='relu', padding='same'))
model.add(Flatten())

model.add(Dense(1024, activation='relu'))
model.add(Dense(512, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(256, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(2, activation='softmax'))

adam = Adam(learning_rate=LEARNING_RATE, beta_1=0.9, beta_2=0.999, decay=0.0, amsgrad=False)
sgdm = SGD(learning_rate=LEARNING_RATE, momentum=0.9)

model.compile(loss=tf.keras.losses.sparse_categorical_crossentropy, optimizer=sgdm, metrics=['accuracy'])

monitor = EarlyStopping(monitor='val_loss', patience=10, verbose=1, mode='min')
checkpointer = ModelCheckpoint(filepath='dnn/model_9/best_weights.keras', verbose=0, save_best_only=True)

model.fit(X_train_reshaped, y_train, validation_data=(X_test_reshaped, y_test), batch_size=BATCH_SIZE, callbacks=[monitor, checkpointer], verbose=2, epochs=EPOCHS)


pred = model.predict(X_test_reshaped)
pred = np.argmax(pred, axis=1)

y_true = y_test

score = metrics.accuracy_score(y_true, pred)
print(f'Accuracy: {score}')

f1 = metrics.f1_score(y_true, pred, average='weighted')
print(f'Average F1: {f1}')

print(metrics.classification_report(y_true, pred))

cm = confusion_matrix(y_true, pred)
print(cm)

plt.figure()
plot_confusion_matrix(cm, ['0', '1'])
plt.show()

plot_roc(pred, y_true)


In [ ]:
model.summary()

In [ ]:
from tensorflow.keras.models import load_model
from sklearn.metrics import f1_score
from sklearn import metrics
from sklearn.metrics import confusion_matrix

X_test = pd.read_csv('dataset/cleaned_dataset/test_feature_analysis.csv')
y_test = X_test['label']
X_test.drop(columns=['id', 'label'], inplace=True)
X_test_pad = np.pad(X_test.values, 
                      ((0, 0), (0, 14*14 - len(X_test.columns))),
                      mode='constant',
                      constant_values=0)
X_test_reshaped = X_test_pad.reshape(-1, 14, 14, 1)

model = load_model('dnn/model_8/best_weights.keras')
pred = model.predict(X_test_reshaped)
pred = np.argmax(pred, axis=1)

y_true = y_test

score = metrics.accuracy_score(y_true, pred)
print(f'Accuracy: {score}')

f1 = metrics.f1_score(y_true, pred, average='weighted')
print(f'Average F1: {f1}')

print(metrics.classification_report(y_true, pred))

cm = confusion_matrix(y_true, pred)
print(cm)

plt.figure()
plot_confusion_matrix(cm, ['Normal', 'Attack'])
plt.show()

plot_roc(pred, y_true)